# M4 - Phosphene simulation

**NTH bootcamp · Module 4**

The interactive companion page `M4-phosphene-simulation.html` lets you move
sliders and watch phosphenes change shape. This notebook does the same
thing in Python, against the *real* [`dynaphos`](https://github.com/neuralcodinglab/dynaphos)
library, and then takes one step further: it drives the simulator from
the output of computer-vision models you already met in M1 and M2 (YOLO
segmentation, depth estimation, DeepGaze gaze prediction, open-vocabulary
detection). The goal is to see - with your own eyes - how the *upstream
representation* chosen by the engineer changes what a prosthesis user
would actually perceive.

1. Phosphene basis (single phosphene, populations)
2. Image → phosphenes (the classical forward pass)
3. Temporal dynamics & rastering (why sequential stimulation dims the percept)
4. **AI-driven stimulus** (YOLO, depth, DeepGaze gaze, open-vocab)
5. Dynamic stimulus loop (a moving scene through the simulator)

Exercises are tagged **`[easy]`**, **`[intermediate]`**, or **`[challenge]`**.
The challenges are genuinely hard; do not feel obliged to finish them in
the guided hour.

## 0 · Setup

Required packages:

```bash
pip install numpy scipy scikit-image opencv-python matplotlib torch dynaphos ultralytics
```

The cell below imports everything, downloads `bus.jpg` and writes a public-domain
astronaut photo (scikit-image) if they aren't already in `assets/`, picks the best
available compute backend (GPU when present, else CPU), and defines a small
`show()` helper used throughout. Run it once.

In [ ]:
# Install all packages this notebook needs. Run once per kernel; skip if already installed.
# DeepGaze III (used in §4.3) ships via the matthias-k/DeepGaze git repo; scipy +
# scikit-image are its helpers and also provide the astronaut demo image used in §2.
%pip install -q numpy scipy scikit-image opencv-python matplotlib torch dynaphos ultralytics
%pip install -q git+https://github.com/matthias-k/DeepGaze.git


In [ ]:
import os, sys, json, urllib.request, tempfile, textwrap, warnings
from pathlib import Path
import numpy as np
import cv2
import matplotlib.pyplot as plt
import torch

ASSETS = Path('assets')
ASSETS.mkdir(exist_ok=True)

# bus.jpg - same image M1 uses (Ultralytics demo image, redistributable)
BUS = ASSETS / 'bus.jpg'
if not BUS.exists():
    try:
        from ultralytics.utils import ASSETS as _UA   # ships bus.jpg, no network needed
        import shutil; shutil.copy(_UA / 'bus.jpg', BUS)
        print(f'copied {BUS} from ultralytics assets')
    except Exception:
        urllib.request.urlretrieve('https://ultralytics.com/images/bus.jpg', BUS)
        print(f'downloaded {BUS}')

# astronaut.png - scikit-image's public-domain demo portrait (NASA, Eileen Collins).
# A face + high-contrast structure makes it the cleanest target for the edge demo in
# §2.2 and for DeepGaze in §4.3 (the same image M2 uses).
PORTRAIT = ASSETS / 'astronaut.png'
if not PORTRAIT.exists():
    try:
        from skimage import data
        import imageio.v3 as iio
        iio.imwrite(PORTRAIT, data.astronaut())     # generated locally, no network
        print(f'wrote {PORTRAIT}')
    except Exception as e:
        print('astronaut image unavailable (a synthetic blob will be used instead):', e)

print('versions:', 'numpy', np.__version__, '· cv2', cv2.__version__, '· torch', torch.__version__)

In [ ]:
# Pick the best available compute backend. GPU by default; CPU fallback.
if torch.cuda.is_available():
    DEVICE = 'cuda'
elif getattr(torch.backends, 'mps', None) and torch.backends.mps.is_available():
    DEVICE = 'mps'      # Apple Silicon
else:
    DEVICE = 'cpu'
print('using device:', DEVICE)


In [ ]:
def show(*imgs, titles=None, cmap='gray', figsize=None, cols=None):
    '''Plot one or more images side-by-side. BGR images are auto-converted to RGB.'''
    if len(imgs) == 1 and isinstance(imgs[0], (list, tuple)):
        imgs = imgs[0]
    n = len(imgs)
    cols = cols or n
    rows = int(np.ceil(n / cols))
    figsize = figsize or (4*cols, 4*rows)
    fig, axes = plt.subplots(rows, cols, figsize=figsize, squeeze=False)
    for i, ax in enumerate(axes.flat):
        ax.axis('off')
        if i >= n: continue
        im = imgs[i]
        if hasattr(im, 'detach'):           # torch tensor
            im = im.detach().cpu().numpy()
        if im.ndim == 3 and im.shape[2] == 3:
            ax.imshow(cv2.cvtColor(im.astype(np.uint8), cv2.COLOR_BGR2RGB))
        else:
            vmax = im.max() if im.dtype != np.uint8 else 255
            ax.imshow(im, cmap=cmap, vmin=0, vmax=vmax if vmax > 0 else 1)
        if titles and i < len(titles):
            ax.set_title(titles[i], fontsize=10)
    plt.tight_layout(); plt.show()


We pin a copy of `dynaphos`'s `config/params.yaml` to a local file so the
rest of the notebook is reproducible. These are the same defaults the
HTML page is built against (rheobase 23.9 µA, current_spread 675 µA/mm²,
view angle 16°, fps 35, cortical dipole k=17.3 / a=0.75 / b=120). All
numbers come from the dynaphos repo. We tell dynaphos to use GPU 0 when
CUDA is available; otherwise its internal device picker falls back to
CPU.


In [ ]:
PARAMS_YAML = r'''run:
  resolution: [256, 256]
  view_angle: 16
  origin: [0, 0]
  min_angle: 0.001
  fps: 35
  gpu: 0
  print_stats: False
  seed: 42
  dtype: float32
  use_gaussian_lut: False
  batch_size: 0
display:
  screen_resolution: [1920, 1080]
  screen_diagonal: 13.3
  dist_to_screen: 600
sampling:
  sampling_method: receptive_fields
  RF_size: 0.5
  stimulus_scale: 1.0e-4
cortex_model:
  model: dipole
  k: 17.3
  a: 0.75
  b: 120
  alpha: 0.95
  dropout_rate: 0.0
  noise_scale: 0.0
temporal_dynamics:
  trace_increase_rate: 13.95528162
  activation_decay_per_second: 0.00012340980408667956
  trace_decay_per_second: 0.99949191
size:
  size_equation: sqrt
  MD: 0.7
  I_half: 40
  radius_to_sigma: 0.5
  current_spread: 675.0e-6
thresholding:
  use_threshold: True
  activation_threshold: 9.141886e-08
  activation_threshold_sd: 0.0
  rheobase: 23.9e-6
default_stim:
  pw_default: 170.0e-6
  freq_default: 300
  relative_stim_duration: 1.0
brightness_saturation:
  use_brightness_saturation: True
  slope_brightness: 1.9152642500946816e+7
  cps_half: 1.057631e-07
gabor:
  gabor_filtering: False
  gamma: 0.5'''

PARAMS_PATH = ASSETS / 'params.yaml'
PARAMS_PATH.write_text(PARAMS_YAML)

# Patch the device hint based on the DEVICE we picked above. dynaphos reads
# params['run']['gpu'] internally and uses it to decide where to put tensors.
import re
if DEVICE != 'cuda':
    txt = PARAMS_PATH.read_text()
    txt = re.sub(r'^(\s*gpu:).*$', r'\1 -1', txt, flags=re.MULTILINE)
    PARAMS_PATH.write_text(txt)
print('wrote', PARAMS_PATH)


## 1 · Phosphene basis

Every phosphene is a **2-D Gaussian** sitting on the visual field. Its
*centre* comes from the retinotopic map (eccentricity + polar angle);
its *size* is set jointly by the current and the local **cortical
magnification** $M(r)$ — the millimetres of cortex devoted to one
degree of visual angle. The fovea has high magnification, so the same
current spreads over fewer degrees and produces a small, sharp dot. The
periphery has low magnification, so the same current paints a much
larger blob.

$$\sigma(\text{deg}) \;=\; s \cdot \frac{\sqrt{I/K}}{M(r)}, \qquad
  M(r) \;=\; \frac{k\,(b-a)}{(r+a)(r+b)}$$

with $s$ = `radius_to_sigma`, $K$ = `current_spread`. We will build a
population of electrodes and visualise this directly.


In [ ]:
from dynaphos.utils import load_params, Map
from dynaphos.cortex_models import (
    get_visual_field_coordinates_probabilistically,
    get_cortical_magnification,
)
from dynaphos.simulator import GaussianSimulator

params = load_params(str(PARAMS_PATH))
rng = np.random.default_rng(params['run']['seed'])

# Foveated population: probability of placing an electrode is proportional to M(r).
N = 600
coords = get_visual_field_coordinates_probabilistically(params, N, rng=rng)
sim = GaussianSimulator(params, coords)
print(f'simulator has {len(coords)} electrodes; phosphenes rendered as '
      f'{tuple(params["run"]["resolution"])} pixel image, '
      f'{params["run"]["view_angle"]}° wide field')


In [ ]:
# Render the whole population once — fire every electrode at the same current.
amp = torch.full((len(coords),), 80e-6, **sim.data_kwargs)   # 80 µA/electrode, on the sim's device & dtype
sim.reset()
for _ in range(60):
    phosphenes = sim(amp)
phosphenes = phosphenes.detach().cpu().numpy()
show(phosphenes, titles=[f'{len(coords)} phosphenes, 80 µA each (foveated layout)'],
     figsize=(5, 5))


The dynaphos library ships a probabilistic foveated sampler
(`get_visual_field_coordinates_probabilistically`) but no parameterised
uniform-grid sampler in the visual field. We build one ourselves with a
tiny helper that wraps the `Map` class.


In [ ]:
def make_uniform_visual_field(side: int, view_angle: float) -> Map:
    '''Side×side electrodes spaced evenly inside [-view_angle/2, +view_angle/2].'''
    hemi = view_angle / 2
    step = view_angle / side
    xs = np.linspace(-hemi + step/2, hemi - step/2, side)
    ys = np.linspace(-hemi + step/2, hemi - step/2, side)
    xx, yy = np.meshgrid(xs, ys)
    return Map(x=xx.ravel(), y=yy.ravel())


### Exercise 1.1 — phosphene size grows with eccentricity `[easy]`

We move a single electrode out along the horizontal meridian and watch two
things change together:

- **Position** — the phosphene shifts away from fixation.
- **Size** — the same 80 µA current covers a larger area because cortical
  magnification $M(r)$ decreases with eccentricity: $\sigma \propto 1/M(r)$.

Instead of an interactive slider (which needs `ipywidgets` and doesn't render
in every environment, Colab included), we render one phosphene per eccentricity
and save the whole sweep as an animated GIF in
`assets/eccentricity_phosphene.gif`. Complete the render loop:

1. `single_coord = Map(x=np.array([r_deg]), y=np.array([0.0]))`
2. `s = GaussianSimulator(params, single_coord)`
3. `amp = torch.full((1,), 80e-6, **s.data_kwargs)`
4. `s.reset()`; run 60 frames; append `ph.detach().cpu().numpy()` to `frames_11`.

The GIF-writing boilerplate below is already done for you.

In [ ]:
import matplotlib.animation as animation
from IPython.display import Image as IPyImage

eccentricities = np.arange(0.5, 7.6, 0.25)   # degrees along the horizontal meridian

# TODO: render one phosphene per eccentricity into frames_11.
#   For each r_deg: Map(x=[r_deg], y=[0.0]) -> GaussianSimulator -> 80 µA, 60 frames,
#   then append ph.detach().cpu().numpy().
frames_11 = []
for r_deg in eccentricities:
    pass  # replace with your code

# --- GIF boilerplate (given) — turns frames_11 into assets/eccentricity_phosphene.gif ---
fig, ax = plt.subplots(figsize=(4, 4))
ax.axis('off')
im = ax.imshow(frames_11[0], cmap='gray', vmin=0, vmax=1)
ttl = ax.set_title('')

def _update(i):
    im.set_data(frames_11[i])
    ttl.set_text(f'eccentricity = {eccentricities[i]:.2f}°')
    return im, ttl

anim = animation.FuncAnimation(fig, _update, frames=len(frames_11), interval=120)
GIF_PATH = ASSETS / 'eccentricity_phosphene.gif'
anim.save(GIF_PATH, writer=animation.PillowWriter(fps=8))
plt.close(fig)
print('saved', GIF_PATH)
IPyImage(filename=str(GIF_PATH))


### Exercise 1.2 — foveated vs uniform electrode layout `[intermediate]`

With the same N electrodes, two layout strategies give very different percepts.
Build both and render each basis map side-by-side.

1. Foveated: `get_visual_field_coordinates_probabilistically(params, N, rng=rng)`.
2. Uniform: `make_uniform_visual_field(side, params['run']['view_angle'])`
   with `side = int(round(np.sqrt(N)))`.
3. Wrap each in `GaussianSimulator(params, coords)`.
4. Fire every electrode at 80 µA for **60 frames**, then render. Plot side-by-side.
5. Which layout gives finer detail at the centre? Which at the periphery?


In [ ]:
# your code here
# N = 600; side = int(round(np.sqrt(N)))
# coords_fov = get_visual_field_coordinates_probabilistically(params, N, rng=rng)
# coords_uni = make_uniform_visual_field(side, params['run']['view_angle'])
# Wrap each in GaussianSimulator(params, coords_...).
# Build the current tensor with the simulator's device/dtype:
#   amp = torch.full((len(coords_...),), 80e-6, **sim_....data_kwargs)
# Run each simulator for 60 frames before rendering.


## 2 · Image → phosphenes (forward pass)

This is the engineer's job: turn an image into a set of stimulation
amplitudes, one per electrode. The simulator's `sample_stimulus(image)`
does this by sampling the image intensity at each electrode's receptive
field. Then `simulator(amplitudes)` runs the physics and renders the
phosphene image.

The signature you'll use over and over:

```python
amplitudes = simulator.sample_stimulus(image, rescale=True)   # (N_electrodes,)
phosphenes = simulator(amplitudes)                            # (H, W), in [0, 1]
```

Input `image` is a `(H, W)` uint8 or float array at the resolution
declared in the params file (256×256 here). `rescale=True` maps the
image's [0, max] range to a reasonable stim amplitude band.


In [ ]:
RES = tuple(params['run']['resolution'])   # (W, H) — 256, 256

def draw_pattern(name: str) -> np.ndarray:
    '''Return a 256×256 grayscale uint8 image for one of the abstract presets.'''
    img = np.zeros(RES[::-1], dtype=np.uint8)
    h, w = img.shape
    if name == 'square_disc':
        cv2.rectangle(img, (60, 60), (160, 160), 255, -1)
        cv2.circle(img, (190, 190), 40, 255, -1)
    elif name == 'letter_E':
        cv2.putText(img, 'E', (40, 210), cv2.FONT_HERSHEY_SIMPLEX, 8.0, 255, 18, cv2.LINE_AA)
    elif name == 'grating':
        for x in range(0, w, 16):
            cv2.rectangle(img, (x, 0), (x+8, h), 255, -1)
    elif name == 'diagonal':
        cv2.line(img, (20, 20), (w-20, h-20), 255, 6, cv2.LINE_AA)
    else:
        raise ValueError(name)
    return img

# Rebuild a fresh simulator that we'll use for §2-§4. Foveated, 1000 electrodes
# so the abstract shapes have something to bind to.
N = 1000
coords = get_visual_field_coordinates_probabilistically(params, N, rng=np.random.default_rng(0))
sim = GaussianSimulator(params, coords)

def render(image_uint8: np.ndarray) -> np.ndarray:
    '''image_uint8 -> phosphene render (H, W) float in [0, 1].'''
    sim.reset()
    amp = sim.sample_stimulus(image_uint8, rescale=True)
    for _ in range(60):
        out = sim(amp)
    return out.detach().cpu().numpy()

target = draw_pattern('square_disc')
show(target, render(target), titles=['target', 'phosphene render'], cols=2, figsize=(8, 4))


### Exercise 2.1 — abstract patterns survive coarse sampling `[easy]`

Loop over the four preset patterns and render each through the
simulator. Display a 2×4 grid: top row = targets, bottom row = phosphene
renders. Which ones stay recognisable, which fall apart?

> Hint: the patterns are `['square_disc', 'letter_E', 'grating', 'diagonal']`.
> Build two lists (`targets`, `renders`) and call
> `show(*(targets + renders), titles=..., cols=4)`.


In [ ]:
# your code here
# Hint:
#   names = ['square_disc', 'letter_E', 'grating', 'diagonal']
#   targets = [draw_pattern(n) for n in names]
#   renders = [render(t) for t in targets]
#   show(*(targets + renders), titles=..., cols=4)


### Exercise 2.2 - natural photos: blobs, then edges `[intermediate]`

Now try a real photo. First push the astronaut portrait through the
simulator raw (just resized to 256x256, grayscale). You should get a
featureless bright blob - the silhouette of the helmet and shoulders.
Then preprocess with `cv2.Canny` first, and pass the edge map through the
same simulator.

1. Load `PORTRAIT` with `cv2.imread(str(PORTRAIT), cv2.IMREAD_GRAYSCALE)`. If
   the file is missing, fall back to a synthetic blob:
   `face = np.zeros(RES[::-1], np.uint8); cv2.ellipse(face, (128,128), (60,80), 0, 0, 360, 255, -1)`.
2. Resize to `RES`, render. Plot input + render.
3. `edges = cv2.Canny(cv2.GaussianBlur(face, (5,5), 1.4), 50, 150)`. Render
   the edges instead. Plot input + edges + edge-render.
4. Which carries more recognisable structure through the prosthesis - the
   raw intensity or the edges?

In [ ]:
face = cv2.imread(str(PORTRAIT), cv2.IMREAD_GRAYSCALE)
if face is None:
    face = np.zeros(RES[::-1], np.uint8)
    cv2.ellipse(face, (128, 128), (60, 80), 0, 0, 360, 255, -1)
face = cv2.resize(face, RES)

raw_render = render(face)
edges = cv2.Canny(cv2.GaussianBlur(face, (5, 5), 1.4), 50, 150)
edge_render = render(edges)

show(face, raw_render, edges, edge_render,
     titles=['astronaut', 'raw → phosphenes', 'Canny edges', 'edges → phosphenes'],
     cols=4, figsize=(14, 4))

## 3 · Temporal dynamics & rastering

A real implant cannot fire every electrode at once. Safety limits cap the
total charge injected per unit time, so high-density arrays **raster**:
the electrodes are split into *timing groups* that switch on one group per
frame, cycling through all groups several times a second. Only a fraction
of the array is ever live at any instant.

That has a perceptual cost. `dynaphos` carries a temporal **activation**
state per electrode - a leaky integrator that charges while the electrode
is driven and decays when it is not. An electrode that is on only 1 frame
in `G` never lets its activation build to the level it would reach under
constant drive, so the phosphene is **dimmer**. How much dimmer depends on
the race between the raster cycle and the activation decay:

* **Group count `G`** is the main brightness knob. With a decay fast
  relative to the cycle, mean brightness falls roughly as **1/G**.
* **Decay rate.** If activation decays slowly, the integrator bridges the
  off-frames and rastering barely dims at all (Exercise 3.3).
* **Raster pattern** (which electrodes share a group) does *not* change the
  time-averaged brightness - over a full cycle every electrode fires once
  regardless. It changes the *instantaneous* spatial appearance and flicker.
  Kasowski & Beyeler (2025) show a **checkerboard** pattern, which maximises
  the spatial separation of simultaneously-active electrodes, is perceived
  more clearly than row/column or random patterns.

> Kasowski, J. M. & Beyeler, M. (2025). *Simulated prosthetic vision confirms
> checkerboard as an effective raster pattern for epiretinal implants.*
> arXiv:2501.02084.

We work at a moderately fast activation decay (`0.1 / s`) so the 1/G law is
clean, then return to the dynaphos default in Exercise 3.3 to see the
integrator defeat it.

In [ ]:
# A dedicated raster simulator. ~400 electrodes is enough to see spatial
# patterns while staying fast on CPU. We bump the activation decay to 0.1/s
# (dynaphos default is ~1.2e-4/s, essentially no decay) so rastering's
# brightness cost is visible rather than bridged by the integrator.
import copy
raster_params = copy.deepcopy(params)
raster_params['temporal_dynamics']['activation_decay_per_second'] = 0.1

coords_r = get_visual_field_coordinates_probabilistically(
    raster_params, 400, rng=np.random.default_rng(7))
sim_r = GaussianSimulator(raster_params, coords_r)
xs_r, ys_r = coords_r.cartesian
print(f'raster simulator: {sim_r.num_phosphenes} electrodes, '
      f"activation decay {raster_params['temporal_dynamics']['activation_decay_per_second']}/s")


def assign_groups(xs, ys, pattern, n_groups, rng):
    '''Assign each electrode to one of `n_groups` timing groups by a spatial rule.
    Returns an int array of group ids in [0, n_groups).'''
    n = len(xs)
    if pattern == 'all_on' or n_groups == 1:
        return np.zeros(n, dtype=int)
    if pattern == 'random':
        return rng.integers(0, n_groups, n)
    if pattern in ('horizontal', 'vertical'):
        key = ys if pattern == 'horizontal' else xs   # sweep top->bottom / left->right
        order = np.argsort(key)
        g = np.empty(n, dtype=int)
        g[order] = (np.arange(n) * n_groups) // n
        return g
    if pattern == 'checkerboard':
        # quantile bins in x and y, then (col+row) mod G -> neighbours land in
        # different groups (maximal spatial separation per frame).
        qx = np.quantile(xs, np.linspace(0, 1, n_groups + 1)[1:-1])
        qy = np.quantile(ys, np.linspace(0, 1, n_groups + 1)[1:-1])
        return (np.digitize(xs, qx) + np.digitize(ys, qy)) % n_groups
    raise ValueError(pattern)


def run_raster(sim, amp_full, groups, n_groups, frames=120, keep_frames=()):
    '''Drive `sim` with `amp_full`, but each frame only the electrodes whose
    group == (t % n_groups) are active. Returns (brightness_per_frame, kept_frames).'''
    sim.reset()
    dev = sim.data_kwargs['device']
    g_t = torch.as_tensor(groups, device=dev)
    bright = np.zeros(frames)
    kept = {}
    for t in range(frames):
        mask = (g_t == (t % n_groups)).to(amp_full.dtype)
        phos = sim(amp_full * mask)
        bright[t] = float(phos.detach().mean())
        if t in keep_frames:
            kept[t] = phos.detach().cpu().numpy()
    return bright, kept


# The stimulus we raster: the whole bus, sampled once into per-electrode currents.
bgr_r = cv2.imread(str(BUS))
gray_r = cv2.resize(cv2.cvtColor(bgr_r, cv2.COLOR_BGR2GRAY),
                    tuple(raster_params['run']['resolution']))
amp_full = sim_r.sample_stimulus(gray_r, rescale=True)
print('per-electrode currents ready:', tuple(amp_full.shape))

In [ ]:
# Showcase: all-on vs a 5-group checkerboard raster. Left = phosphene field at a
# few frames; right = mean brightness over time. The rastered field flickers and
# sits well below the all-on brightness.
fps = raster_params['run']['fps']
rng = np.random.default_rng(0)
G = 5
groups_all = assign_groups(xs_r, ys_r, 'all_on', 1, rng)
groups_chk = assign_groups(xs_r, ys_r, 'checkerboard', G, rng)

picks = (0, 1, 2)
b_all, frames_all = run_raster(sim_r, amp_full, groups_all, 1, frames=90, keep_frames=picks)
b_chk, frames_chk = run_raster(sim_r, amp_full, groups_chk, G, frames=90, keep_frames=picks)

fig = plt.figure(figsize=(11, 4.2))
gs = fig.add_gridspec(2, len(picks) + 2)
for j, t in enumerate(picks):
    ax = fig.add_subplot(gs[0, j]); ax.imshow(frames_all[t], cmap='gray', vmin=0, vmax=1)
    ax.set_title(f'all-on, frame {t}', fontsize=8); ax.axis('off')
    ax = fig.add_subplot(gs[1, j]); ax.imshow(frames_chk[t], cmap='gray', vmin=0, vmax=1)
    ax.set_title(f'checker G=5, frame {t}', fontsize=8); ax.axis('off')
axc = fig.add_subplot(gs[:, len(picks):])
tt = np.arange(len(b_all)) / fps
axc.plot(tt, b_all, label='all-on (G=1)', color='C0')
axc.plot(tt, b_chk, label=f'checkerboard (G={G})', color='C1')
axc.axhline(b_all[30:].mean(), color='C0', ls=':', alpha=0.6)
axc.axhline(b_chk[30:].mean(), color='C1', ls=':', alpha=0.6)
axc.set_xlabel('time (s)'); axc.set_ylabel('mean phosphene brightness')
axc.set_title('rastering dims the percept'); axc.legend()
plt.tight_layout(); plt.show()
print(f'steady-state brightness  all-on={b_all[30:].mean():.4f}  '
      f'checker G=5={b_chk[30:].mean():.4f}  ratio={b_chk[30:].mean()/b_all[30:].mean():.2f}')

### Exercise 3.1 - the brightness-vs-group-count law `[easy]`

Sweep the number of timing groups `G` over `[1, 2, 3, 5, 8]` using a
checkerboard assignment, and plot the steady-state mean brightness
(averaged over the second half of the run) as a fraction of the all-on
(`G = 1`) brightness. Overlay the `1/G` reference curve.

You should see the measured ratio track `1/G` closely - each electrode is
live `1/G` of the time, and at this decay rate the activation integrator
has mostly emptied between visits, so brightness scales with duty cycle.

In [ ]:
# Exercise 3.1 - your turn
# Hint:
#   groups_list = [1, 2, 3, 5, 8]
#   base = steady-state brightness of the all-on run (assign_groups(..., 'all_on', 1, rng))
#   for each G: grp = assign_groups(xs_r, ys_r, 'checkerboard', G, rng)
#               b, _ = run_raster(sim_r, amp_full, grp, G)
#               ratio = b[60:].mean() / base
#   Plot ratio vs G, and overlay [1/G for G in groups_list] as a reference line.

# your code here

### Exercise 3.2 - raster patterns: same brightness, different appearance `[intermediate]`

Fix `G = 5` and compare four patterns: `horizontal`, `vertical`,
`checkerboard`, `random`. For each, run the raster and plot the mean
brightness over time on shared axes; also show a single mid-run frame of
each so you can see the spatial structure of what is lit at one instant.

What you should find: the **time-averaged** brightness curves sit on top of
one another - over a full cycle every electrode fires exactly once no
matter how you group them, so the pattern does not change mean brightness.
What changes is the *instantaneous* frame: `horizontal`/`vertical` light a
band at a time, `checkerboard` scatters the live electrodes apart, `random`
looks like noise. Kasowski & Beyeler (2025) found the checkerboard's spatial
separation makes the integrated percept the easiest to read - a perceptual
win the mean-brightness curve cannot show.

In [ ]:
# Exercise 3.2 - your turn
# Hint:
#   patterns = ['horizontal', 'vertical', 'checkerboard', 'random']; G = 5
#   For each pattern: grp = assign_groups(xs_r, ys_r, pattern, G, rng)
#                     b, kept = run_raster(sim_r, amp_full, grp, G, keep_frames=(61,))
#   Plot every pattern's brightness-over-time on one axis (they overlap!), then show
#   one mid-run frame (kept[61]) per pattern so the spatial layout differs.

# your code here

### Exercise 3.3 - the integrator can hide rastering `[intermediate]`

Everything above used a fast activation decay (`0.1 / s`). Repeat the
group-count sweep from 3.1 at the **dynaphos default** decay
(`~1.2e-4 / s`, essentially no decay within a run) as well, and plot both
brightness-vs-`G` curves together.

At the default decay the activation integrator barely leaks between an
electrode's visits, so it tops up to nearly the all-on level even when the
electrode is live only part of the time - the `G = 2` point hardly dims at
all, and only large `G` (long off-gaps) start to lose brightness. The
dimming you get from rastering is therefore not a fixed property of the
implant; it is set by the race between the raster cycle and how fast the
tissue forgets.

In [ ]:
# Exercise 3.3 - your turn
# Hint:
#   Write sweep(decay) that rebuilds a sim with that activation_decay_per_second
#   (copy.deepcopy(params); set p['temporal_dynamics']['activation_decay_per_second']),
#   then returns the checkerboard brightness/all-on ratio for G in [1,2,3,5,8].
#   Call it twice: sweep(0.1) and sweep(params['temporal_dynamics']['activation_decay_per_second']).
#   Plot both curves + the 1/G reference. The slow one should barely dim at small G.

# your code here

## 4 · AI-driven stimulus

Up to here, the stimulus was either an abstract shape or a raw photo
intensity sampled at each electrode. Real prostheses don't have to work
that way - the input to `sim.sample_stimulus(...)` can be **anything**
you can compute from the camera. This section drives the same simulator
from the outputs of four computer-vision models and compares the resulting
phosphene renders.

* **4.1** YOLOv8n-seg - class-weighted object masks
* **4.2** MiDaS - monocular depth, near = bright
* **4.3** DeepGaze III - gaze-gated edges: keep edges where the user would look
* **4.4** YOLO-World - open-vocabulary detection from a free-text class list

We run all of them on `bus.jpg` (the M1 image) so the renders are
visually comparable.

In [ ]:
# Load and resize bus.jpg to the simulator's resolution.
bgr = cv2.imread(str(BUS))
scene_gray = cv2.resize(cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY), RES)
scene_bgr  = cv2.resize(bgr, RES)
show(scene_bgr, render(scene_gray),
     titles=['scene (raw intensity input)', 'phosphenes from raw intensity'],
     cols=2, figsize=(8, 4))


### YOLO segmentation → phosphenes

YOLOv8n-seg returns one binary mask per detection. We collapse them into
a single "objects vs background" activation map and feed *that* in
instead of raw pixels. Things that aren't objects (sky, pavement) go
dark; people, vehicles, etc. light up.


In [ ]:
yolo_ok = False
try:
    from ultralytics import YOLO
    yolo_model = YOLO('yolov8n-seg.pt')
    yolo_res = yolo_model.predict(str(BUS), device=DEVICE, verbose=False)[0]
    yolo_ok = True
    print(f'YOLO ran on {DEVICE}. {len(yolo_res.boxes)} detections.')
except Exception as e:
    print('YOLO unavailable, will fall back to brightness threshold:', e)

def get_yolo_masks_resized(out_shape):
    '''Return (masks (N,H,W) uint8, class_ids (N,), names dict).'''
    if not yolo_ok:
        # Fallback: split bright-vs-dim regions of the scene as a 1-class "mask"
        m = (scene_gray > 110).astype(np.uint8)
        return m[None, :, :], np.array([0]), {0: 'bright'}
    raw = yolo_res.masks.data.cpu().numpy()
    H, W = out_shape
    masks = np.stack([
        (cv2.resize(m, (W, H), interpolation=cv2.INTER_LINEAR) > 0.5).astype(np.uint8)
        for m in raw
    ])
    cls = yolo_res.boxes.cls.cpu().numpy().astype(int)
    return masks, cls, yolo_res.names

masks, cls, names = get_yolo_masks_resized(RES[::-1])
union = (np.any(masks.astype(bool), axis=0).astype(np.uint8) * 255)
show(scene_bgr, union, render(union),
     titles=['scene', 'YOLO object union', 'phosphenes from YOLO union'],
     cols=3, figsize=(12, 4))


### Exercise 4.1 — class-weighted activation `[easy]`

Not every class is equally informative for a prosthesis user. Build a
single `(H, W)` float map where each pixel takes the **largest** weight
of any object covering it, using
`weights = {'person': 1.0, 'bus': 0.5, 'skateboard': 0.2}`. Multiply by
255, cast to uint8, and feed through `render(...)`. People should be the
brightest blobs in the resulting phosphene render.

Variables already in scope: `masks` `(N, H, W)`, `cls` `(N,)`,
`names` `dict[int,str]`.


In [ ]:
# your code here
# Hint:
#   weights = {'person': 1.0, 'bus': 0.5, 'skateboard': 0.2}
#   canvas = np.zeros(RES[::-1], dtype=np.float32)
#   for m, c in zip(masks, cls):
#       w = weights.get(names[int(c)], 0.0)
#       canvas = np.maximum(canvas, w * m.astype(np.float32))
#   weighted = (canvas * 255).astype(np.uint8)
#   show(...); render(weighted)


### Exercise 4.2 — monocular depth → phosphenes `[intermediate]`

A prosthesis built for **navigation** cares about *what's close*, not
*what's bright*. Use the MiDaS small monocular-depth model to estimate
relative depth, then make a "near = bright" activation map.

1. Load MiDaS via `torch.hub`:
   `midas = torch.hub.load('intel-isl/MiDaS', 'MiDaS_small').to(DEVICE).eval()`
2. Get its preprocessing transform:
   `transform = torch.hub.load('intel-isl/MiDaS', 'transforms').small_transform`
3. Run on `cv2.cvtColor(scene_bgr, cv2.COLOR_BGR2RGB)`, getting a `(H, W)`
   depth map. (MiDaS returns inverse depth: larger = closer.)
4. Normalise to `[0, 255]` uint8 (already near = bright since it's inverse
   depth), resize to `RES`, render.
5. Display the depth map alongside its phosphene render.

If MiDaS isn't available, fall back to `cv2.applyColorMap`-free
brightness inversion as a stand-in (`(255 - scene_gray)`).


In [ ]:
# your code here
# Hint (MiDaS):
#   midas = torch.hub.load('intel-isl/MiDaS', 'MiDaS_small', trust_repo=True).to(DEVICE).eval()
#   transform = torch.hub.load('intel-isl/MiDaS', 'transforms', trust_repo=True).small_transform
#   rgb = cv2.cvtColor(scene_bgr, cv2.COLOR_BGR2RGB)
#   inp = transform(rgb).to(DEVICE)
#   with torch.no_grad():
#       pred = midas(inp)
#       pred = torch.nn.functional.interpolate(pred.unsqueeze(1), size=RES[::-1],
#                                              mode='bicubic', align_corners=False).squeeze()
#   depth = pred.cpu().numpy()
#   depth_u8 = normalise depth to uint8
# Fallback: depth = (255.0 - scene_gray).astype(np.float32)


In [ ]:
# DeepGaze III setup for §4.3 (ported from M2). One-time ~80 MB weight download.
import sys, types
import torch.nn.functional as F
from scipy.ndimage import zoom
from scipy.special import logsumexp

# deepgaze_pytorch eagerly imports OpenAI CLIP (a variant we don't use); stub it.
sys.modules.setdefault('clip', types.ModuleType('clip'))
import deepgaze_pytorch

# MIT1003 center bias - reuse M2's copy if present, else fetch it (~8 MB).
CB_PATH = ASSETS / 'centerbias_mit1003.npy'
if not CB_PATH.exists():
    m2_cb = Path('../M2-deepgaze-and-gaze/assets/centerbias_mit1003.npy')
    if m2_cb.exists():
        import shutil; shutil.copy(m2_cb, CB_PATH)
    else:
        urllib.request.urlretrieve(
            'https://github.com/matthias-k/DeepGaze/releases/download/v1.0.0/centerbias_mit1003.npy',
            CB_PATH)
print('center bias:', CB_PATH)

print('loading DeepGaze III weights (one-time ~80 MB) ...')
dg_model = deepgaze_pytorch.DeepGazeIII(pretrained=True).to(DEVICE).eval()
DG_SIZE = 768   # DeepGaze sees the image at this resolution internally (see M2 for why)


def gaze_heatmap(bgr_image):
    '''Free-viewing DeepGaze III gaze probability for a BGR image.
    Returns a (H, W) float map (H, W = RES) that sums to 1.'''
    rgb = cv2.cvtColor(cv2.resize(bgr_image, RES), cv2.COLOR_BGR2RGB)
    H, W = rgb.shape[:2]
    img_t = torch.tensor(rgb.transpose(2, 0, 1)[None].astype(np.float32)).to(DEVICE)

    templ = np.load(CB_PATH)
    cb = zoom(templ, (H / templ.shape[0], W / templ.shape[1]), order=0, mode='nearest')
    cb = cb - logsumexp(cb)
    cb_t = torch.tensor(cb[None].astype(np.float32)).to(DEVICE)

    img_dg = F.interpolate(img_t, size=(DG_SIZE, DG_SIZE), mode='bilinear', align_corners=False)
    cb_dg = F.interpolate(cb_t[:, None], size=(DG_SIZE, DG_SIZE), mode='bilinear',
                          align_corners=False)[:, 0]
    cb_dg = cb_dg - torch.logsumexp(cb_dg.reshape(1, -1), dim=1).view(1, 1, 1)

    # free viewing: no fixation history -> NaN slots deactivate the scanpath encoder.
    scale = DG_SIZE / W
    nan = float('nan')
    xs_h = np.array([[nan, nan, nan, (W // 2) * scale]], dtype=np.float32)
    ys_h = np.array([[nan, nan, nan, (H // 2) * scale]], dtype=np.float32)
    xh = torch.tensor(xs_h[:, dg_model.included_fixations]).to(DEVICE)
    yh = torch.tensor(ys_h[:, dg_model.included_fixations]).to(DEVICE)
    with torch.no_grad():
        log_d = dg_model(img_dg, cb_dg, xh, yh)
    p = F.adaptive_avg_pool2d(torch.exp(log_d), (H, W))
    p = p / p.sum()
    return p.squeeze().detach().cpu().numpy()

print('gaze_heatmap() ready')

### Exercise 4.3 - gaze-gated edges → phosphenes `[intermediate]`

In M2 you met **DeepGaze III**, a model that predicts where a human would
look in an image. Here we use that prediction to decide *which edges are
worth spending phosphenes on*. The idea: a Canny edge map of the whole
scene has far more contour than a sparse implant can show, so we **keep
edges bright where the user is likely to look and dim them (never to zero)
elsewhere**.

The cell above this one loaded DeepGaze and defined `gaze_heatmap(bgr)`,
which returns a `(H, W)` probability map that sums to 1.

1. Compute `heat = gaze_heatmap(scene_bgr)` and normalise it to `[0, 1]`.
2. Build the Canny edge map of `scene_gray` (float in `[0, 1]`).
3. Make a gating weight `w = FLOOR + (1 - FLOOR) * heat_norm` with
   `FLOOR = 0.15` - so the least-attended edges keep 15 % of their strength
   rather than vanishing.
4. `gated = edges * w`; scale to uint8 and `render(...)` it.
5. Show scene, heatmap, gated edges, and the phosphene render side by side.

Compare with the plain edge render from §2.2: the gaze-gated version spends
its limited phosphene budget on the bus front and the pedestrians instead of
the pavement texture.

In [ ]:
# Exercise 4.3 - your turn
# Hint:
#   edges = cv2.Canny(cv2.GaussianBlur(scene_gray,(5,5),1.4), 50,150).astype(np.float32)/255
#   heat = gaze_heatmap(scene_bgr); heat_norm = (heat-heat.min())/(np.ptp(heat)+1e-9)
#   FLOOR = 0.15
#   gate  = FLOOR + (1-FLOOR)*heat_norm          # in [FLOOR, 1] - never zero
#   gated = edges * gate
#   gated_u8 = (gated/(gated.max()+1e-9)*255).astype(np.uint8)
#   show(scene_bgr, heat_norm, gated, render(gated_u8), titles=[...], cols=4)

# your code here

### Exercise 4.4 — open-vocabulary prompts `[challenge]`

YOLOv8 sees only the 80 COCO classes. **YOLO-World** accepts a free-text
class list and finds whatever you ask for — "stop sign", "crosswalk",
"door". The challenge is authoring the class list so the resulting
phosphene render is **legible**: pick few large objects, not many small
ones.

```python
from ultralytics import YOLO
ovw = YOLO('yolov8s-world.pt')                              # downloads ~50 MB on first run
ovw.set_classes(['bus', 'person', 'door'])
ov_res = ovw.predict(str(BUS), device=DEVICE, verbose=False)[0]
```

1. Get the boxes from `ov_res.boxes.xyxy.cpu().numpy()`. (YOLO-World
   gives bounding boxes; for segmentation masks you'd swap in
   `yoloe-11s-seg.pt`.)
2. Build a binary mask: for each box, paint a filled rectangle of 255
   into a zeros canvas of shape `RES[::-1]`.
3. Render and display.

Try a few class lists. What happens to the render as you add more, smaller objects?


In [ ]:
# your code here
# Hint:
#   from ultralytics import YOLO
#   ovw = YOLO('yolov8s-world.pt')
#   ovw.set_classes(['bus', 'person', 'door'])
#   ov_res = ovw.predict(str(BUS), device=DEVICE, verbose=False)[0]
#   boxes = ov_res.boxes.xyxy.cpu().numpy()
#   Build a uint8 canvas of shape RES[::-1] and draw filled rectangles for each box
#   (remember to rescale the boxes from the original image size to RES).
#   render(canvas) and show().


## 5 · Dynamic stimulus loop

So far §3 watched what one electrode does over time *with a constant
stimulus*, and §4 watched what happens with *one image*. The real world
has both at once: the camera moves, objects move, and adaptation
accumulates while the content underneath the electrodes changes.

The interactive HTML page added a "drifting image" loop after round-2
feedback to make this visible. The challenge below is the Python
equivalent.


### Exercise 5.1 — pan a scene through the simulator `[challenge]`

Take any §4 stimulus (the YOLO union is the simplest), pan it
horizontally by 2 px per frame for 60 frames, and run the *same*
simulator across the sequence **without** resetting between frames.

1. Pick a stimulus image, e.g. `stim = union.copy()` (uint8, shape `RES[::-1]`).
2. Build a `frames` list. For each `t` in `range(60)`:
   * `shifted = np.roll(stim, shift=t*2, axis=1)`
   * `amp = sim.sample_stimulus(shifted, rescale=True)`
   * `phos = sim(amp)` — **do not** call `sim.reset()` inside the loop
   * append `phos.detach().cpu().numpy()` to `frames`
3. Display every 10th frame in a row, plus a brightness-over-time plot.

Compare to a reset-every-frame run (call `sim.reset()` inside the loop).
Which is more faithful to a real implant?


In [ ]:
# your code here
# Hint:
#   def run_pan(reset_each_frame):
#       sim.reset()
#       frames, bright = [], []
#       stim = union.copy()
#       for t in range(60):
#           if reset_each_frame: sim.reset()
#           shifted = np.roll(stim, shift=t*2, axis=1)
#           amp = sim.sample_stimulus(shifted, rescale=True)
#           phos = sim(amp)
#           frames.append(phos.detach().cpu().numpy())
#           bright.append(float(phos.detach().mean()))
#       return frames, np.array(bright)
#   Run twice (reset True / False), show every 10th frame, plot brightness vs time.


## 6 · Bring your own implant *(optional)*

Up to here the electrode layout was hard-coded inside this notebook.
[`vimplant2`](https://antonio-lozano.github.io/vimplant2/) is a browser
tool (no install) that lets you place implant patches on a real cortical
surface and export the resulting visual-field coverage as CSV. Drop the
file next to this notebook and feed it into the same `dynaphos`
simulator from §2.

We ship one example, `vimplant2-rfs-example.csv`, generated with the
same Schwartz log-polar wedge-dipole the HTML §02.1 preview uses. Once
the pipeline is clear, replace it with your own export.


**Expected CSV columns** — exactly what vimplant2's *Export RFs (CSV)*
button writes:

| column | meaning |
|---|---|
| `source_app` | always `web_explorer` for vimplant2 web exports |
| `dataset` | which retinotopic atlas the RFs came from (NHP / human) |
| `prf_source` | which subject / parcellation produced the RFs |
| `implant_id` | which implant the row belongs to (multi-implant scenes) |
| `electrode_index` | per-implant index, 0-based |
| `x_deg`, `y_deg` | visual-field coordinates in degrees |
| `polar_deg`, `ecc_deg` | same point, polar form |

`load_vimplant2_csv` reads the file, sanity-checks the schema, and
returns an `(N, 2)` array of `(x_deg, y_deg)` ready to feed into
`GaussianSimulator`.


In [ ]:
import csv
import warnings

def load_vimplant2_csv(path: str) -> np.ndarray:
    '''Read a vimplant2 RF-export CSV and return (N, 2) (x_deg, y_deg) coords.'''
    required = {'x_deg', 'y_deg'}
    coords, seen_app = [], None
    with open(path, newline='', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        missing = required - set(reader.fieldnames or [])
        if missing:
            raise ValueError(f'CSV is missing required column(s): {missing}')
        for row in reader:
            if seen_app is None:
                seen_app = (row.get('source_app') or '').strip()
            try:
                coords.append((float(row['x_deg']), float(row['y_deg'])))
            except (TypeError, ValueError):
                continue
    if seen_app and seen_app != 'web_explorer':
        warnings.warn(
            f"CSV's source_app is {seen_app!r}, expected 'web_explorer'. "
            'Proceeding anyway, but double-check the file came from vimplant2.'
        )
    if not coords:
        raise ValueError(f'No rows with finite x_deg/y_deg in {path}')
    return np.asarray(coords, dtype=np.float32)


In [ ]:
EXAMPLE_CSV = Path('vimplant2-rfs-example.csv')
coords_vimplant = load_vimplant2_csv(str(EXAMPLE_CSV))
ecc_vimplant = np.hypot(coords_vimplant[:, 0], coords_vimplant[:, 1])
print(f'loaded {len(coords_vimplant)} electrodes from {EXAMPLE_CSV.name}')
print(f'  mean ecc {ecc_vimplant.mean():.2f}°  ·  max ecc {ecc_vimplant.max():.2f}°')


### Exercise 6.1 — visualise the layout `[easy]`

Scatter the loaded electrodes on a square axes spanning ±8°. Overlay
eccentricity guide rings at 2°, 5°, and 10° so a peripheral cluster is
obviously peripheral, and mark the fovea at the origin.

> Hint: `plt.scatter(coords_vimplant[:, 0], coords_vimplant[:, 1], s=14)`.
> For each `e` in `[2, 5, 10]`, add `plt.Circle((0, 0), e, fill=False)`
> via `ax.add_patch(...)`. Set `ax.set_aspect('equal')`.


In [ ]:
# your code here
# Hint:
#   fig, ax = plt.subplots(figsize=(5, 5))
#   ax.scatter(coords_vimplant[:, 0], coords_vimplant[:, 1], s=14)
#   for e in [2, 5, 10]:
#       ax.add_patch(plt.Circle((0, 0), e, fill=False, color='C3'))
#   ax.set_aspect('equal'); ax.set_xlim(-8, 8); ax.set_ylim(-8, 8)


### Exercise 6.2 — render through dynaphos `[intermediate]`

Feed the vimplant2 coordinates straight into a fresh `GaussianSimulator`
— same call signature as §2, just with the layout coming from the CSV
instead of the procedural foveated sampler. Render the same
`target = draw_pattern('square_disc')` through both simulators and
display them side by side.

> Hint: wrap the loaded coords as `Map(x=coords_vimplant[:, 0], y=coords_vimplant[:, 1])`
> and build `sim_v = GaussianSimulator(params, coords_map)`. Then
> `sim_v.reset(); ph = sim_v(sim_v.sample_stimulus(target, rescale=True)).detach().cpu().numpy()`.


In [ ]:
# your code here
# Hint:
#   coords_map = Map(x=coords_vimplant[:, 0], y=coords_vimplant[:, 1])
#   sim_v = GaussianSimulator(params, coords_map)
#   target = draw_pattern('square_disc')
#   sim_v.reset()
#   ph_v = sim_v(sim_v.sample_stimulus(target, rescale=True)).detach().cpu().numpy()
#   show(target, render(target), ph_v, titles=[...], cols=3)


**Your own implant.** Open
[vimplant2](https://antonio-lozano.github.io/vimplant2/), place a patch
wherever you like, click **Export RFs (CSV)**, save the file alongside
this notebook (e.g. `vimplant2-rfs-mine.csv`), and re-run the two cells
above with `path='vimplant2-rfs-mine.csv'`. Every other cell in §6 is
agnostic to where the layout came from.


---

**Done.** You drove the same dynaphos simulator from several different
upstream representations (intensity, edges, YOLO masks, depth, gaze-gated
edges, open-vocab boxes), and you saw how **rastering** - splitting the
electrodes into timing groups that fire in sequence - trades brightness
for safety. Bring whichever stimulus path you found most legible into M5 -
it becomes the input to the decoding-and-closed-loop module.

Module lead: Lefteris & Jorge. Edit this notebook directly; commit your
additions to the bootcamp repo at the end of the day.